# Motorcars

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
%matplotlib inline


## Laden

In [ ]:
df = pd.read_csv('./mtcars.csv')

## Exploration

In [ ]:
df.shape

In [ ]:
df.info()

* Bedeutung der Spalten
  * model = Name
  * mpg = Miles per Gallon
  * cyl = Anzahl der Zylinder
  * displacement = Hubraum
  * hp = PS
  * drat = Achsenübersetzung
  * wt = Gewicht
  * qsec = "Quarter Mile Time"
  * vs = Motortyp, 0=V-Shaped, 1 = normal
  * am = Gangschaltung, 0=automatisch, 1=manuell
  * gear = Anzahl Gänge ohne Rückwärtsgang
  * carb = Anzahl der Vergaser (= carburetors)

* Ergebnis
  * Fast alles numerisch
  * Kandidaten Kategorien
    * cyl, vs, am und gear
    * Model

In [ ]:
df['model'].nunique()

* model ist nicht sinnvoll kategorisierbar, hat ja soviele Werte wie der gesamte Datensatz

In [ ]:
category_candidates = ('cyl', 'vs', 'am', 'gear', 'carb')
for column in category_candidates:
    print(f'{column} has {df[column].nunique()} values')


* Ergebnis
  * cyl, vs, am, gear, carb sind kategorisierbar
  

In [ ]:
for column in category_candidates:
    df[column] = pd.Categorical(df[column])
df.info()    

* Schauen wir uns potenziellen Zusammenhänge an: 

In [ ]:
plots = sns.PairGrid(df)
plots.map(sns.scatterplot)

* oder als Heatmap

In [ ]:
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True)

* Verteilung der Daten

In [ ]:
df.boxplot('mpg')

In [ ]:
df.boxplot('mpg', by='cyl')

In [ ]:
df.boxplot('hp', by='cyl')

In [ ]:
df.boxplot('wt', by='cyl')

In [ ]:
df.hist('mpg')

In [ ]:
df.hist('mpg', 'cyl')

## Analyse

### Deskriptive Statistik

* Übersicht

In [ ]:
df.describe()

* Skewness

In [ ]:
df['mpg'].skew()

* Kurtosis
  * "Schlankheit" im Vergleich zu einer Normalverteilung

In [ ]:
df['mpg'].kurt()

In [ ]:
df['mpg'].plot(kind='density')

## Regressionsanalyse

In [ ]:
df.plot(kind='scatter', x='wt', y='mpg')

* Wir machen nun die Regressionsanalyse
  * Lineare Regression

In [ ]:
from sklearn import linear_model # pip install scikit-learn 

In [ ]:
regression_model = linear_model.LinearRegression()
regression_model.fit(X=pd.DataFrame(df['wt']), y = df['mpg'])
print(f'Achsenabschnitt: {regression_model.intercept_}')
print(f'Steigung: {regression_model.coef_}')

In [ ]:
predicted = regression_model.predict(X=pd.DataFrame(df['wt']))

In [ ]:
residuals = df['mpg'] - predicted
residuals.describe()

In [ ]:
df.plot(kind='scatter', x='wt', y='mpg')
plt.plot(df['wt'].to_numpy(), predicted)

In [ ]:
regression_model.score(X=pd.DataFrame(df['wt']), y=df['mpg'])

In [ ]:
import scipy.stats as stats

In [ ]:
stats.probplot(residuals, dist='norm', plot=plt)

* Polynome Regression
  * 2. Ordnung = Statt Ausgleichsgerade nehmen wir eine Parabel

In [ ]:
poly_model = linear_model.LinearRegression()
x_axis = pd.DataFrame([df['wt'], df['wt']**2]).T
poly_model.fit(X=x_axis, y=df['mpg'])
print(f'Achenabschnitt: {poly_model.intercept_}')
print(f'Koeffizienten: {poly_model.coef_}')
print(f"Score: {poly_model.score(x_axis, y=df['mpg'])}")


In [ ]:
df.plot(kind='scatter', x='wt', y='mpg')
poly_range = np.arange(1.5, 5.5,0.01)
predicted = poly_model.predict(pd.DataFrame([poly_range, poly_range**2]).T)
plt.plot(poly_range, predicted)


   * Hinzufügen weiterer Potenzen, in den meisten Fällen sinnlos, da nie physikalisch begründbar

In [ ]:
useless_poly_model = linear_model.LinearRegression()
x_axis = pd.DataFrame([
    df['wt'], 
    df['wt']**2,
    df['wt']**3,
    df['wt']**4,
    df['wt']**5,
    df['wt']**6,
    df['wt']**7,
    df['wt']**8,
    df['wt']**9,
    ]).T
useless_poly_model.fit(X=x_axis, y=df['mpg'])
print(f'Achenabschnitt: {useless_poly_model.intercept_}')
print(f'Koeffizienten: {useless_poly_model.coef_}')
print(f"Score: {useless_poly_model.score(x_axis, y=df['mpg'])}")

In [ ]:
df.plot(kind='scatter', x='wt', y='mpg')
useless_predicted = useless_poly_model.predict(pd.DataFrame([
    poly_range, 
    poly_range**2,
    poly_range**3,
    poly_range**4,
    poly_range**5,
    poly_range**6,
    poly_range**7,
    poly_range**8,
    poly_range**9,
    ]).T)
plt.plot(poly_range, useless_predicted)


* Multiple Linear Regression

In [ ]:
multi_reg_model = linear_model.LinearRegression()

# Include squared terms
poly_predictors = pd.DataFrame([df["wt"],
                                df["hp"],
                                df["wt"]**2,
                                df["hp"]**2]).T

# Train the model using the mtcars data
multi_reg_model.fit(X = poly_predictors, 
                    y = df["mpg"])
print(f'Achenabschnitt: {multi_reg_model.intercept_}')
print(f'Koeffizienten: {multi_reg_model.coef_}')
print(f"Score: {multi_reg_model.score(poly_predictors, y=df['mpg'])}")
